In [1]:
# Cell 1: what's inside OpenFake? Reads only file lists and small text columns, never the images.
!pip -q install pyarrow "huggingface_hub<2.0"

import collections
import pyarrow.parquet as pq
from huggingface_hub import HfApi, HfFileSystem

REPO = "ComplexDataLab/OpenFake"

# If the dataset turns out to need a login, we'll add a token here. Try without first.
api = HfApi()
fs = HfFileSystem()

# 1) Which files exist?
files = api.list_repo_files(REPO, repo_type="dataset")
parquets = sorted(f for f in files if f.endswith(".parquet"))
print("Total files:", len(files), "| parquet files:", len(parquets))
print("Parquet files per folder/split:",
      collections.Counter("/".join(f.split("/")[:-1]) + " : " + f.split("/")[-1].split("-")[0]
                          for f in parquets))
print("Non-parquet files:", [f for f in files if not f.endswith(".parquet")])

# 2) Open ONE test file and look at its columns (schema only, a few KB)
first_test = next(f for f in parquets if f.startswith("core/test"))
pf = pq.ParquetFile(fs.open(f"datasets/{REPO}/{first_test}"))
print("\nFile:", first_test)
print("Rows:", pf.metadata.num_rows, "| row groups:", pf.metadata.num_row_groups)
print(pf.schema_arrow)

# 3) Read only the small text columns (skip the image bytes) from the first row group
text_cols = [n for n in pf.schema_arrow.names
             if pf.schema_arrow.field(n).type not in ("binary", "large_binary")
             and "image" not in n.lower()]
print("\nText columns:", text_cols)
sample = pf.read_row_group(0, columns=text_cols).to_pandas()
print(sample.head(5).to_string())

Total files: 647 | parquet files: 643
Parquet files per folder/split: Counter({'core : train': 608, 'core : validation': 15, 'core : test': 13, 'reddit : test': 5, 'corrections : prompt_mapping_fix.parquet': 1, 'corrections : v2_removed_rows.parquet': 1})
Non-parquet files: ['.gitattributes', 'README.md', 'corrections/v2_repair_summary.json', 'prompt-image_bank.csv']

File: core/test-00000-of-00013.parquet
Rows: 7031 | row groups: 48
image: struct<bytes: binary, path: string>
  child 0, bytes: binary
  child 1, path: string
prompt: string
label: string
model: string
type: string
release_date: string
row_id: string
image_sha256: string
prompt_kind: string
prompt_model: string
prompt_revision: string
prompt_version: string
-- schema metadata --
huggingface: '{"info":{"features":{"image":{"_type":"Image"},"prompt":{"d' + 510
openfake_repair: '{"schema_version":"openfake-hub-repair-v1"}'

Text columns: ['prompt', 'label', 'model', 'type', 'release_date', 'row_id', 'prompt_kind', 'prompt_mo

In [2]:
# Cell 2: which generators are in OpenFake, and how many images of each? (text columns only)
import pandas as pd

def read_meta(path):
    pf = pq.ParquetFile(fs.open(f"datasets/{REPO}/{path}"))
    t = pf.read(columns=["label", "model", "type", "release_date"]).to_pandas()
    t["file"] = path
    return t

train_files = [f for f in parquets if f.startswith("core/train")]
val_files   = [f for f in parquets if f.startswith("core/validation")]

# A spread of train files (beginning, middle, end) plus 2 validation files
picked = train_files[::120][:6] + val_files[:2]
meta = pd.concat([read_meta(f) for f in picked], ignore_index=True)
print("Rows read:", len(meta), "from", len(picked), "files")

print("\nLabel balance:")
print(meta["label"].value_counts())

print("\nType values:")
print(meta["type"].value_counts())

print("\nFake images per generator (with release date):")
fakes = meta[meta["label"] == "fake"]
print(fakes.groupby("model").agg(count=("model", "size"), released=("release_date", "first"))
           .sort_values("count", ascending=False).to_string())

print("\nReal images per source:")
print(meta[meta["label"] == "real"]["model"].value_counts().to_string())

print("\nDoes one file mix many generators? Distinct generators per file:")
print(fakes.groupby("file")["model"].nunique().to_string())

Rows read: 27796 from 8 files

Label balance:
label
real    13941
fake    13855
Name: count, dtype: int64

Type values:
type
real                11155
base                 9728
finetune             1996
Base                 1615
image                1483
lora                  740
Fine-tune             530
video                 307
base-refined          189
LoRA                   30
photo                  14
base-watermarked        9
Name: count, dtype: int64

Fake images per generator (with release date):
                                   count    released
model                                               
sd-3.5                              1317     2024-01
sd-2.1                              1234     2024-01
flux.1-dev                          1052     2024-01
sdxl-epic-realism                    579     2024-01
midjourney-6                         541     2023-12
flux.1-schnell                       435     2024-01
gpt-image-1                          408     2025-04
sd-1.5-dream

In [3]:
# Cell 3: look at real images from one chunk: format, size, download speed
import io, time
from PIL import Image

f = train_files[0]
pf = pq.ParquetFile(fs.open(f"datasets/{REPO}/{f}"))

t0 = time.time()
rg = pf.read_row_group(0, columns=["image", "label", "model"]).to_pylist()
secs = time.time() - t0
mb = sum(len(r["image"]["bytes"]) for r in rg) / 1e6
print(f"One chunk: {len(rg)} images, {mb:.0f} MB in {secs:.1f}s  ->  {mb/secs:.1f} MB/s")
print(f"Estimate for 25,000 images: {25000/len(rg)*secs/60:.0f} min if we keep every image in each chunk")

rows = []
for r in rg:
    b = r["image"]["bytes"]
    im = Image.open(io.BytesIO(b))
    rows.append({"label": r["label"], "model": r["model"], "format": im.format,
                 "w": im.size[0], "h": im.size[1], "kb": len(b) // 1024})
d = pd.DataFrame(rows)

print("\nFile format by label:")
print(d.groupby("label")["format"].value_counts().to_string())
print("\nSize by label (median):")
print(d.groupby("label")[["w", "h", "kb"]].median().to_string())
print("\nLong side by label (min / median / max):")
d["long"] = d[["w", "h"]].max(axis=1)
print(d.groupby("label")["long"].agg(["min", "median", "max"]).to_string())

One chunk: 69 images, 129 MB in 2.4s  ->  54.5 MB/s
Estimate for 25,000 images: 14 min if we keep every image in each chunk

File format by label:
label  format
fake   PNG       17
       JPEG       7
real   JPEG      43
       PNG        2

Size by label (median):
            w       h     kb
label                       
fake    960.0  1024.0  674.0
real   1300.0  1562.0  982.0

Long side by label (min / median / max):
       min  median   max
label                   
fake   512  1024.0  1408
real   160  1562.0  8192


In [4]:
# Cell 4: build the training and validation sets (same treatment for real and fake)
import os, io, random, shutil, collections, time
from concurrent.futures import ProcessPoolExecutor
from PIL import Image

OUT = "/tmp/of"
CROP = 384                                   # Community Forensics input size: crop, never squash
WA_QUALITY, WA_MAX_SIDE = 80, 1600           # measured from your forwarded files
HELD_OUT = {"seedream-v4.0", "seedream-v4.5", "qwen-image"}   # never trained on: unseen-generator test
CAP = {"train": 250, "val": 60}              # max fake images per generator (forces variety)
REAL_CAP = {"train": 4000, "val": 750}       # max real images per source (laion, pexels)
print("Free disk in /tmp: %.0f GB | CPUs: %d" % (shutil.disk_usage("/tmp").free / 1e9, os.cpu_count()))

def jpeg(img, q):
    b = io.BytesIO()
    img.save(b, "JPEG", quality=q, subsampling=2)
    return Image.open(io.BytesIO(b.getvalue())).convert("RGB")

def whatsapp(img):
    """Simulated forward: long side capped at 1600, JPEG quality 80, 4:2:0 (matches your files)."""
    w, h = img.size
    s = WA_MAX_SIDE / max(w, h)
    if s < 1:
        img = img.resize((round(w * s), round(h * s)), Image.BICUBIC)
    return jpeg(img, WA_QUALITY)

def crop(img, rng=None):
    w, h = img.size
    if rng is None:                          # validation: centre crop (same spot for both versions)
        x, y = (w - CROP) // 2, (h - CROP) // 2
    else:                                    # training: random spot
        x, y = rng.randint(0, w - CROP), rng.randint(0, h - CROP)
    return img.crop((x, y, x + CROP, y + CROP))

def process(job):
    b, label, model, split, idx = job
    rng = random.Random(idx)
    try:
        img = Image.open(io.BytesIO(b)).convert("RGB")
    except Exception:
        return []
    if min(img.size) < 512:                  # skip thumbnails
        return []
    if split == "train":
        if rng.random() < 0.3:               # random shrink for BOTH classes, so "resized" means nothing
            s = rng.uniform(0.5, 0.9)
            w, h = img.size
            if min(w, h) * s >= CROP:
                img = img.resize((round(w * s), round(h * s)), Image.BICUBIC)
        r = rng.random()
        if r < 0.2:
            versions = [("none", img)]
        elif r < 0.6:
            versions = [("jpeg", jpeg(img, rng.randint(65, 95)))]
        else:
            versions = [("whatsapp", whatsapp(img))]
    else:                                    # validation: every image twice, clean and forwarded
        versions = [("none", img), ("whatsapp", whatsapp(img))]
    rows = []
    for aug, im in versions:
        if min(im.size) < CROP:
            continue
        path = f"{OUT}/{split}/{idx}_{aug}.png"
        crop(im, rng if split == "train" else None).save(path)
        rows.append({"path": path, "label": int(label == "fake"), "model": model,
                     "split": split, "aug": aug, "idx": idx})
    return rows

def collect(files, split, pool):
    os.makedirs(f"{OUT}/{split}", exist_ok=True)
    taken = collections.Counter()
    futures, idx, t0 = [], (0 if split == "train" else 10_000_000), time.time()
    for fi, f in enumerate(files):
        pf = pq.ParquetFile(fs.open(f"datasets/{REPO}/{f}"))
        for g in range(pf.metadata.num_row_groups):
            meta = pf.read_row_group(g, columns=["label", "model"]).to_pylist()   # tiny
            want = []
            for i, m in enumerate(meta):
                key = m["model"]
                if m["label"] == "fake":
                    if split == "train" and key in HELD_OUT:
                        continue
                    if taken[key] >= CAP[split]:
                        continue
                elif taken["real:" + key] >= REAL_CAP[split]:
                    continue
                taken[key if m["label"] == "fake" else "real:" + key] += 1
                want.append(i)
            if not want:
                continue                     # nothing needed from this chunk: skip downloading it
            imgs = pf.read_row_group(g, columns=["image"]).column("image").to_pylist()
            for i in want:
                futures.append(pool.submit(process, (imgs[i]["bytes"], meta[i]["label"],
                                                     meta[i]["model"], split, idx)))
                idx += 1
            pending = [x for x in futures[-1000:] if not x.done()]
            while len(pending) > 300:        # don't let downloads run far ahead of processing (RAM)
                pending[0].exception()           # waits for it to finish (errors handled later)
                pending = [x for x in pending if not x.done()]
        print(f"[{split}] file {fi + 1}/{len(files)} done, {len(futures)} images queued, "
              f"{(time.time() - t0) / 60:.1f} min")
    rows, failed = [], 0
    for fut in futures:
        try:
            rows += fut.result()
        except Exception:                    # one broken image shouldn't stop the whole build
            failed += 1
    print(f"[{split}] finished: {len(rows)} crops saved, {failed} images failed")
    return pd.DataFrame(rows)

with ProcessPoolExecutor(max_workers=os.cpu_count()) as pool:
    val_df = collect(val_files[:2], "val", pool)
    train_df = collect(train_files[::60][:10], "train", pool)

index = pd.concat([train_df, val_df], ignore_index=True)
index.to_csv("/kaggle/working/of_index.csv", index=False)

print("\nImages saved:")
print(index.groupby(["split", "label"]).size().to_string())
print("\nGenerators in train:", train_df[train_df.label == 1].model.nunique(),
      "| in val:", val_df[val_df.label == 1].model.nunique())
print("Held-out generator images in val:", val_df[val_df.model.isin(HELD_OUT)].idx.nunique())
print("Train augmentation mix:", train_df.aug.value_counts().to_dict())
print("Disk used: %.1f GB" % (sum(os.path.getsize(p) for p in index.path) / 1e9))

Free disk in /tmp: 1149 GB | CPUs: 4
[val] file 1/2 done, 3393 images queued, 4.4 min
[val] file 2/2 done, 4931 images queued, 6.6 min
[val] finished: 8948 crops saved, 0 images failed


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[train] file 1/10 done, 3958 images queued, 4.6 min
[train] file 2/10 done, 7414 images queued, 9.7 min


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ComplexDataLab/OpenFake/resolve/main/core/train-00006-of-00032-00006.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ComplexDataLab/OpenFake/resolve/main/core/train-00006-of-00032-00006.parquet
Retrying in 1s [Retry 1/5].


[train] file 3/10 done, 10658 images queued, 14.6 min


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ComplexDataLab/OpenFake/resolve/main/core/train-00009-of-00032-00009.parquet
Retrying in 1s [Retry 1/5].
'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ComplexDataLab/OpenFake/resolve/main/core/train-00009-of-00032-00009.parquet
Retrying in 2s [Retry 2/5].
/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1136: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


[train] file 4/10 done, 13502 images queued, 19.2 min
[train] file 5/10 done, 14399 images queued, 21.8 min
[train] file 6/10 done, 14924 images queued, 23.9 min
[train] file 7/10 done, 14934 images queued, 23.9 min
[train] file 8/10 done, 15394 images queued, 26.1 min
[train] file 9/10 done, 15825 images queued, 28.4 min


'The read operation timed out' thrown while requesting GET https://huggingface.co/datasets/ComplexDataLab/OpenFake/resolve/main/core/train-00028-of-00032-00008.parquet
Retrying in 1s [Retry 1/5].


[train] file 10/10 done, 16208 images queued, 30.8 min
[train] finished: 13866 crops saved, 0 images failed

Images saved:
split  label
train  0        5757
       1        8109
val    0        2160
       1        6788

Generators in train: 71 | in val: 58
Held-out generator images in val: 177
Train augmentation mix: {'jpeg': 5662, 'whatsapp': 5451, 'none': 2753}
Disk used: 4.2 GB


In [5]:
# Cell 5: load the Community Forensics model and measure it BEFORE fine-tuning
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "timm==1.0.15"], check=True)
    import timm
import numpy as np, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from sklearn.metrics import roc_auc_score

device = "cuda"
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # the authors' normalisation

def load_commfor():
    net = timm.create_model("vit_small_patch16_384.augreg_in21k_ft_in1k", pretrained=False)
    net.head = nn.Linear(net.head.in_features, 1)
    state = load_file(hf_hub_download("OwensLab/commfor-model-384", "model.safetensors"))
    net.load_state_dict({k.removeprefix("vit."): v for k, v in state.items()})
    return net.to(device)

class Crops(Dataset):
    def __init__(self, df, train=False):
        self.df = df.reset_index(drop=True)
        steps = [transforms.RandomHorizontalFlip()] if train else []
        self.tf = transforms.Compose(steps + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open(r.path).convert("RGB")), torch.tensor(float(r.label))

@torch.no_grad()
def scores(net, df):
    net.eval()
    out = []
    for x, _ in DataLoader(Crops(df), batch_size=128, num_workers=4):
        with torch.autocast("cuda", dtype=torch.float16):
            out.append(torch.sigmoid(net(x.to(device)).float()).squeeze(1).cpu())
    return torch.cat(out).numpy()

def report(net, title):
    v = val_df.copy()
    v["p"] = scores(net, v)
    print(f"\n=== {title} ===")
    res = {}
    for aug in ["none", "whatsapp"]:
        part = v[v.aug == aug]
        res[aug] = float(roc_auc_score(part.label, part.p))
        held = part[(part.label == 0) | part.model.isin(HELD_OUT)]
        res[aug + "_unseen"] = float(roc_auc_score(held.label, held.p))
        print(f"{aug:9} AUC all: {res[aug]:.3f} | AUC unseen generators: {res[aug + '_unseen']:.3f} | n={len(part)}")
    # per-generator AUC on the WhatsApp version (each generator's fakes vs all real)
    wa = v[v.aug == "whatsapp"]
    wa_real = wa[wa.label == 0]
    per = {m: roc_auc_score(np.r_[np.zeros(len(wa_real)), np.ones(len(g))], np.r_[wa_real.p, g.p])
           for m, g in wa[wa.label == 1].groupby("model") if len(g) >= 20}
    per = pd.Series(per).sort_values()
    print("Weakest generators after fake-WhatsApp:", per.head(6).round(3).to_dict())
    for name in ["gpt-image-1", "nano-banana-2", "imagen-4.0", "dalle-3"]:
        if name in per:
            print(f"  {name}: {per[name]:.3f}")
    return res, v

net = load_commfor()
before, _ = report(net, "Community Forensics, before fine-tuning")

model.safetensors: reconstructing file:   0%|          |  0.00B / 87.3MB            

model.safetensors: downloading bytes:           |  0.00B            


=== Community Forensics, before fine-tuning ===
none      AUC all: 0.936 | AUC unseen generators: 0.889 | n=4474
whatsapp  AUC all: 0.928 | AUC unseen generators: 0.879 | n=4474
Weakest generators after fake-WhatsApp: {'flux.2-klein-4b': 0.295, 'flux-mvc5000': 0.679, 'flux-realism': 0.74, 'gpt-image-1': 0.751, 'nano-banana-2': 0.781, 'imagen-3.0-002': 0.788}
  gpt-image-1: 0.751
  nano-banana-2: 0.781
  imagen-4.0: 0.789
  dalle-3: 0.999


In [6]:
# Cell 6: fine-tune on OpenFake crops, keep the epoch with the best fake-WhatsApp AUC
import copy, json, time
EPOCHS, LR, BATCH = 3, 1e-5, 32

loader = DataLoader(Crops(train_df, train=True), batch_size=BATCH, shuffle=True,
                    num_workers=4, drop_last=True)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=0.05)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(loader))
scaler = torch.amp.GradScaler()

# Train set is 41% real / 59% fake: weight fakes down so both classes count equally
pos_weight = torch.tensor((train_df.label == 0).sum() / (train_df.label == 1).sum(),
                          dtype=torch.float32, device=device)
loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_auc, best_state, history = 0, None, []
for epoch in range(1, EPOCHS + 1):
    net.train()
    t0, total = time.time(), 0.0
    for step, (x, y) in enumerate(loader):
        x, y = x.to(device), y.to(device)
        with torch.autocast("cuda", dtype=torch.float16):
            loss = loss_fn(net(x).squeeze(1).float(), y)
        opt.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        sched.step()
        total += loss.item()
        if step % 100 == 0:
            print(f"epoch {epoch} step {step}/{len(loader)} loss {loss.item():.3f}")
    print(f"epoch {epoch}: mean loss {total / len(loader):.3f}, {(time.time() - t0) / 60:.1f} min")
    res, v = report(net, f"after epoch {epoch}")
    history.append({"epoch": epoch, "loss": total / len(loader), **res})
    if res["whatsapp"] > best_auc:
        best_auc, best_state, best_v = res["whatsapp"], copy.deepcopy(net.state_dict()), v

# Cut-off: the score that flags only 5% of real validation images as AI
net.load_state_dict(best_state)
real_p = best_v[best_v.label == 0].p
threshold = float(np.quantile(real_p, 0.95))
for aug in ["none", "whatsapp"]:
    part = best_v[best_v.aug == aug]
    caught = (part[part.label == 1].p >= threshold).mean()
    false_alarm = (part[part.label == 0].p >= threshold).mean()
    print(f"{aug:9} at cut-off {threshold:.3f}: fakes caught {caught:.1%}, real flagged {false_alarm:.1%}")

torch.save(best_state, "/kaggle/working/verify_v4.pth")
json.dump({"base": "OwensLab/commfor-model-384", "timm_name": "vit_small_patch16_384.augreg_in21k_ft_in1k",
           "crop": CROP, "mean": MEAN, "std": STD, "threshold": threshold,
           "threshold_rule": "95th percentile of real validation scores (5% false alarms)",
           "held_out": sorted(HELD_OUT), "lr": LR, "epochs": EPOCHS, "history": history,
           "before": before, "train_images": len(train_df), "val_images": len(val_df),
           "version": "commfor-openfake-wa-v4"},
          open("/kaggle/working/v4_config.json", "w"), indent=2)
print("Saved verify_v4.pth and v4_config.json. Best fake-WhatsApp AUC:", round(best_auc, 3))

/tmp/ipykernel_22/648741667.py:28: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


epoch 1 step 0/433 loss 1.041
epoch 1 step 100/433 loss 0.135
epoch 1 step 200/433 loss 0.194
epoch 1 step 300/433 loss 0.276
epoch 1 step 400/433 loss 0.185
epoch 1: mean loss 0.187, 2.7 min

=== after epoch 1 ===
none      AUC all: 0.990 | AUC unseen generators: 0.931 | n=4474
whatsapp  AUC all: 0.970 | AUC unseen generators: 0.885 | n=4474
Weakest generators after fake-WhatsApp: {'qwen-image': 0.785, 'flux.2-klein-4b': 0.788, 'flux-1.1-pro': 0.854, 'gpt-image-1': 0.867, 'nano-banana-2': 0.902, 'flux-realism': 0.908}
  gpt-image-1: 0.867
  nano-banana-2: 0.902
  imagen-4.0: 0.963
  dalle-3: 0.997
epoch 2 step 0/433 loss 0.076
epoch 2 step 100/433 loss 0.024
epoch 2 step 200/433 loss 0.033
epoch 2 step 300/433 loss 0.059
epoch 2 step 400/433 loss 0.065
epoch 2: mean loss 0.084, 2.7 min

=== after epoch 2 ===
none      AUC all: 0.993 | AUC unseen generators: 0.957 | n=4474
whatsapp  AUC all: 0.977 | AUC unseen generators: 0.910 | n=4474
Weakest generators after fake-WhatsApp: {'qwen-im